In [2]:
# This notebook evaluates models using 10 repeated splits with wealth-exclusive features
def enforce_model_dtypes(df):
    df = df.copy()

    # -------- Category columns --------
    category_cols = [
        "hv000CountryCode",
        "hv270WealthIndex",
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
        "v717Occupation",
        "745aHouseOwnership",
    ]

    for col in category_cols:
        if col in df.columns:
            df[col] = df[col].astype("category")

    # -------- Integer columns --------
    int8_cols = [
        "hv220AgeOfHead",
        "hv009FamilySize",
        "hv014NoOfChildren",
        "EnergyPoor",
    ]

    for col in int8_cols:
        if col in df.columns:
            df[col] = df[col].astype("int8")

    # -------- Float columns --------
    float_cols = [
        "hv216HouseSize",
        "HouseholdClusterElevation",
    ]

    for col in float_cols:
        if col in df.columns:
            df[col] = df[col].astype("float64")

       return df


In [ ]:
import pandas as pd
from autogluon.tabular import TabularPredictor
SPLITS_ROOT = r"COMBINE/DATA/PAIRTEST10"

def evaluate_saved_models(country, trad_key, dl_key="FASTAI", n_splits=10):
    rows = []

    for i in range(1, n_splits + 1):
        # test data
        test_path = f"{SPLITS_ROOT}/split{i}_test.parquet"
        test_df = pd.read_parquet(test_path)
         # ---- Enforce dtypes----
        test_df  = enforce_model_dtypes(test_df)

        # saved model folders (based on your structure)
        trad_path = rf"{SPLITS_ROOT}\MODELS\WEALTH_EXCLUSIVE\split{i}\TRAD_{trad_key}"
        dl_path   = rf"{SPLITS_ROOT}\MODELS\WEALTH_EXCLUSIVE\split{i}\DL_{dl_key}"

        # load predictors
        pred_tr = TabularPredictor.load(trad_path)
        pred_dl = TabularPredictor.load(dl_path)

        # evaluate
        m_tr = pred_tr.evaluate(test_df, silent=True)
        m_dl = pred_dl.evaluate(test_df, silent=True)
        
        acc_tr = float(m_tr["accuracy"])
        acc_dl = float(m_dl["accuracy"])
        
        mcc_tr = float(m_tr["mcc"])
        mcc_dl = float(m_dl["mcc"])
        
        balacc_tr = float(m_tr["balanced_accuracy"])
        balacc_dl = float(m_dl["balanced_accuracy"])
        
        f1_tr = float(m_tr["f1"])
        f1_dl = float(m_dl["f1"])

        rows.append({
            "Country": country,
            "Split": i,
            "Traditional_Model": trad_key,
            "DL_Model": dl_key,
        
            "Traditional_Accuracy": acc_tr,
            "DL_Accuracy": acc_dl,
            "Diff_Accuracy": acc_tr - acc_dl,
        
            "Traditional_MCC": mcc_tr,
            "DL_MCC": mcc_dl,
            "Diff_MCC": mcc_tr - mcc_dl,
        
            "Traditional_BalancedAcc": balacc_tr,
            "DL_BalancedAcc": balacc_dl,
        
            "Traditional_F1": f1_tr,
            "DL_F1": f1_dl,
        })

        print(f"{country} split {i}: TRAD={acc_tr:.4f}, DL={acc_dl:.4f}")

    return pd.DataFrame(rows)

In [ ]:
import os
import numpy as np
from scipy.stats import ttest_rel

RESULTS_FOLDER = rf"{SPLITS_ROOT}\RESULTS"
os.makedirs(RESULTS_FOLDER, exist_ok=True)

country_trad = {
    "COMBINE": "GBM",
}

all_df = []
print("\n================ SPLIT-LEVEL RESULTS ================\n")
for country, trad_key in country_trad.items():
    df_country = evaluate_saved_models(country, trad_key, dl_key="FASTAI", n_splits=10)
    all_df.append(df_country)

df_splits = pd.concat(all_df, ignore_index=True)

#print(df_splits)

# ---------- Model Performance ----------
def mean_std(x):
    return f"{x.mean():.3f} ± {x.std():.3f}"

table6_rows = []

for country in df_splits["Country"].unique():
    sub = df_splits[df_splits["Country"] == country]

    ml_model = sub["Traditional_Model"].iloc[0]
    dl_model = sub["DL_Model"].iloc[0]

    table6_rows.append({
        "Country": country,
        "Model": ml_model,

        "Accuracy (mean ± SD)": mean_std(sub["Traditional_Accuracy"]),
        "MCC (mean ± SD)": mean_std(sub["Traditional_MCC"]),
        "Balanced Accuracy (mean ± SD)": mean_std(sub["Traditional_BalancedAcc"]),
        "F1-Score (mean ± SD)": mean_std(sub["Traditional_F1"]),
    })

    table6_rows.append({
        "Country": country,
        "Model": dl_model,

        "Accuracy (mean ± SD)": mean_std(sub["DL_Accuracy"]),
        "MCC (mean ± SD)": mean_std(sub["DL_MCC"]),
        "Balanced Accuracy (mean ± SD)": mean_std(sub["DL_BalancedAcc"]),
        "F1-Score (mean ± SD)": mean_std(sub["DL_F1"]),
    })

print("\n================ AVERAGE RESULTS ================\n")
df_table6 = pd.DataFrame(table6_rows)
df_table6

In [5]:
save_confirm = True   # change to False if don't want to save yet
if save_confirm:
    country_folder = rf"{RESULTS_FOLDER}\{country}"
    os.makedirs(country_folder, exist_ok=True)
    
    df_table6.to_csv(rf"{country_folder}\Table7_combine_best_performing_wealth_exclusive.csv", index=False)
    print("\nFiles saved successfully.")

In [ ]:
from scipy.stats import ttest_rel
import numpy as np

ttest_rows = []

for country in df_splits["Country"].unique():

    sub = df_splits[df_splits["Country"] == country].sort_values("Split")

    ml_model = sub["Traditional_Model"].iloc[0]
    dl_model = sub["DL_Model"].iloc[0]

    # ---------------- ACCURACY ----------------
    trad_acc = sub["Traditional_Accuracy"].to_numpy()
    dl_acc   = sub["DL_Accuracy"].to_numpy()

    diff_acc = trad_acc - dl_acc
    t_acc, p_acc = ttest_rel(trad_acc, dl_acc)

    # ---------------- MCC ----------------
    trad_mcc = sub["Traditional_MCC"].to_numpy()
    dl_mcc   = sub["DL_MCC"].to_numpy()

    diff_mcc = trad_mcc - dl_mcc
    t_mcc, p_mcc = ttest_rel(trad_mcc, dl_mcc)

    # ---------------- FINAL ROW ----------------
    ttest_rows.append({
        "Country": country,
        "Model Comparison": f"{ml_model} vs {dl_model}",

        "Δ Acc": diff_acc.mean(),
        "SD Acc": diff_acc.std(ddof=1),
        "t Acc": t_acc,
        "p Acc": p_acc,

        "Δ MCC": diff_mcc.mean(),
        "SD MCC": diff_mcc.std(ddof=1),
        "t MCC": t_mcc,
        "p MCC": p_mcc,
    })
# ---------- Summary statistics ----------
print("\n================ Summary statistics ================\n")
df_ttest = pd.DataFrame(ttest_rows)
df_ttest


In [7]:
save_confirm = True   # change to False if don't want to save yet
if save_confirm:
    country_folder = rf"{RESULTS_FOLDER}\{country}"
    os.makedirs(country_folder, exist_ok=True)
    
    df_ttest.to_csv(rf"{country_folder}\Table10_combine_best_model_t_test_wealth_exclusive.csv", index=False)
    print("\nFiles saved successfully.")